# Topic Modeling with BERTopic — Parameter Search & Evaluation (SynBio Papers)

Loads the pre-computed **SynBio Papers** embeddings, runs a grid search over
key UMAP/HDBSCAN parameters, evaluates each configuration with **C_v
coherence**, **topic diversity**, and **DBCV**, selects the best model,
optionally reassigns outliers, and saves the results to `assets/topic_models/`.

> **Recommended** — this is the notebook used in the associated publication.

In [1]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live one level below 02-topic_model/, where the aux/ package
# resides; add that folder to the import path.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
from aux.paths import MODELS_DIR, set_seed
from aux.topic_modeling import load_corpus, save_topic_outputs
from aux.evaluation import grid_search

set_seed()

# ── CONFIG: SynBio Papers ──────────────────────────────────────────────────
EMBEDDINGS_FILE = "papers_embeddings.npy"
CORPUS_FILE     = "papers_corpus.txt"
ID_COL          = "id"
PREFIX          = "papers"

PARAM_GRID = {
    "min_cluster_size": [20, 30, 40, 50],
    "umap_n_neighbors": [10, 15, 25],
    "umap_n_components": [5, 10],
}

# Keep HDBSCAN noise labels — some papers may be genuinely off-topic.
REDUCE_OUTLIERS = False

## 1. Load embeddings and corpus

In [3]:
embeddings, corpus = load_corpus(EMBEDDINGS_FILE, CORPUS_FILE)
docs = corpus["text"].tolist()
print(f"Papers: {embeddings.shape[0]:,} docs, {embeddings.shape[1]} dims")

Papers: 19,017 docs, 384 dims


## 2. Grid search

Fit and evaluate a BERTopic model for every parameter combination.

In [4]:
results, best = grid_search(docs, embeddings, PARAM_GRID, label="Papers")
results

[Papers] 1/24  mcs=20, nn=10, nc=5 ... topics=243, C_v=0.6520, div=0.6502, DBCV=0.1836
[Papers] 2/24  mcs=20, nn=10, nc=10 ... topics=228, C_v=0.6532, div=0.6421, DBCV=0.1855
[Papers] 3/24  mcs=20, nn=15, nc=5 ... topics=227, C_v=0.6592, div=0.6515, DBCV=0.1948
[Papers] 4/24  mcs=20, nn=15, nc=10 ... topics=225, C_v=0.6615, div=0.6564, DBCV=0.1981
[Papers] 5/24  mcs=20, nn=25, nc=5 ... topics=221, C_v=0.6537, div=0.6412, DBCV=0.2021
[Papers] 6/24  mcs=20, nn=25, nc=10 ... topics=201, C_v=0.6409, div=0.6338, DBCV=0.1557
[Papers] 7/24  mcs=30, nn=10, nc=5 ... topics=158, C_v=0.6037, div=0.5576, DBCV=0.1984
[Papers] 8/24  mcs=30, nn=10, nc=10 ... topics=154, C_v=0.6067, div=0.5519, DBCV=0.2130
[Papers] 9/24  mcs=30, nn=15, nc=5 ... topics=158, C_v=0.6116, div=0.5766, DBCV=0.1843
[Papers] 10/24  mcs=30, nn=15, nc=10 ... topics=154, C_v=0.6194, div=0.5812, DBCV=0.2086
[Papers] 11/24  mcs=30, nn=25, nc=5 ... topics=144, C_v=0.6059, div=0.5389, DBCV=0.2009
[Papers] 12/24  mcs=30, nn=25, nc=10

,min_cluster_size,n_neighbors,n_components,n_topics,outlier_frac,coherence_cv,diversity,dbcv
0,20,15,10,225,0.3271,0.6615,0.6564,0.1981
1,20,15,5,227,0.3018,0.6592,0.6515,0.1948
2,20,25,5,221,0.3646,0.6537,0.6412,0.2021
3,20,10,10,228,0.2933,0.6532,0.6421,0.1855
4,20,10,5,243,0.2751,0.6520,0.6502,0.1836
5,20,25,10,201,0.3302,0.6409,0.6338,0.1557
6,30,15,10,154,0.3496,0.6194,0.5812,0.2086
7,30,15,5,158,0.3385,0.6116,0.5766,0.1843
8,30,25,10,137,0.3620,0.6085,0.5511,0.1650
9,30,10,10,154,0.3064,0.6067,0.5519,0.2130


In [5]:
print("Best configuration:")
for k in ["min_cluster_size", "n_neighbors", "n_components", "n_topics",
          "coherence_cv", "diversity", "dbcv", "outlier_frac"]:
    print(f"  {k:16s} = {best[k]}")

Best configuration:
  min_cluster_size = 20
  n_neighbors      = 15
  n_components     = 10
  n_topics         = 225
  coherence_cv     = 0.6615
  diversity        = 0.6564
  dbcv             = 0.1981
  outlier_frac     = 0.3271


## 3. Reduce outliers (optional)

HDBSCAN labels documents that fall outside any dense cluster as topic **−1**
(noise). While that is acceptable for the SynBio literature (some papers may be
genuinely off-topic), every iGEM team project is by definition related to
synthetic biology — its text may simply be too short or idiosyncratic to land in
a cluster. BERTopic's `reduce_outliers` (strategy `"embeddings"`, threshold `0`)
reassigns **all** noise documents to their nearest topic by cosine similarity,
without retraining the model.

Controlled by `REDUCE_OUTLIERS` in the config above (enabled for teams,
disabled for papers).

In [6]:
model = best["model"]
topics = list(best["topics"])

if REDUCE_OUTLIERS:
    before = sum(1 for t in topics if t == -1)
    topics = model.reduce_outliers(
        docs, topics, strategy="embeddings", embeddings=embeddings, threshold=0,
    )
    model.update_topics(docs, topics=topics)
    after = sum(1 for t in topics if t == -1)
    print(f"Outliers: {before:,} → {after:,}")
else:
    print("Outlier reduction disabled — keeping HDBSCAN noise labels.")

Outlier reduction disabled — keeping HDBSCAN noise labels.


## 4. Save best model, outputs, and grid-search results

In [7]:
save_topic_outputs(model, corpus, topics, ID_COL, PREFIX)
results.to_csv(MODELS_DIR / f"{PREFIX}_grid_search.txt", sep="\t", index=False)

print(f"Saved → {MODELS_DIR}")
for f in sorted(MODELS_DIR.glob(f"{PREFIX}_*")):
    print(f"  {f.name}")

2026-06-11 15:25:32,231 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Saved → /Users/cristian/Desktop/GitHub/igem-synbio/assets/topic_models
  papers_doc_topics.txt
  papers_grid_search.txt
  papers_topic_info.txt
  papers_topic_model
  papers_topic_names.txt
